In [ ]:
from pathlib import Path

import numpy as np

from cardiac_electrophysiology import components
from cardiac_electrophysiology.utils import visualization

In [2]:
settings = builder.PosteriorBuilderSettings(
    paths=builder.Paths(
        vtu_mesh_path=Path("../data/patient_01/mesh_with_fibers_tags.vtu"),
        xdmf_mesh_path=Path("../data/patient_01/mesh.xdmf"),
        basis_vecs_path=Path("../data/patient_01/basis_vecs.npy"),
        log_file_path=Path("lsbip_logfile.log"),
        ground_truth_path=None,
        noisy_data_path=None,
        fiber_ensemble_path=None,
    ),
    strategies=builder.Strategies(
        ground_truth_strategy="from_mesh",
        mean_strategy="from_ground_truth",
        noisy_data_strategy="from_ground_truth",
    ),
    prior_parameters=builder.PriorParameters(
        kappa=0.05,
        tau=5,
        seed=0,
    ),
    eikonal_parameters=builder.EikonalParameters(
        solver_tolerance=1e-6,
        max_num_iterations=1000,
        max_value=1000,
        initial_site_ind=12650,
        longitudinal_velocity=3,
        transversal_velocity=1,
    ),
    observation_parameters=builder.ObservationParameters(
        num_observations=5000,
        noise_variance=1e-4,
        seed=0,
    ),
    logger_settings=builder.LoggerSettings(
        do_printing=False,
        write_mode="w",
    ),
)

posterior_builder = builder.PosteriorBuilder(settings)
posterior, additional_output = posterior_builder.build(return_additional_data=True)
prior = posterior.prior

In [6]:
sample = prior.generate_sample()
angles = additional_output.angle_transformator.compute_angle_from_parameter(sample) / 2
fibers = additional_output.fiber_transformator.compute_fiber_from_angle(angles)
angles = additional_output.fiber_transformator.compute_angle_from_fiber(fibers)
parameters = additional_output.angle_transformator.compute_parameter_from_angle(angles)
angles = additional_output.angle_transformator.compute_angle_from_parameter(parameters)
visualization.visualize_full_scalar_field(
    mesh=additional_output.pv_mesh,
    scalar_field=angles,
)
np.save("ground_truth_from_prior.npy", fibers)

Widget(value='<iframe src="http://localhost:41671/index.html?ui=P_0x7f218d1c8cd0_3&reconnect=auto" class="pyvi…